In [1]:
# compute silhouette score of ST
# compute ss of ct-SVG
    # average ss

# average ss of ST and ct-SVG on all runs

# Moran's I of all SVGs in ST-orig
# Moran's I of all ct-SVGs in ST-decomp

# compute jaccard of ST

# union all ct-SVG for each cell type
    # compute jaccard

# Average Moran's I of all genes
# Average Moran's I on all runs
# Average jaccard score on all runs

import pandas as pd
import scanpy as sc
import torch

from sklearn.metrics import silhouette_score, silhouette_samples

from pathlib import Path

from hyperspot.preprocessing import common

from hyperspot.metadata import ProjectMetadata
from hyperspot.utils import (
    load_ct_spatial_data,
    load_orig_st_data,
    load_aggr_st_data,
)

/opt/conda/envs/HGCTSVG/lib/python3.12/site-packages/numcodecs/__init__.py:106: DeprecationWarning: crc32c usage is deprecated since numcodecs v0.16.4. It is recommended to install google_crc32c instead.
  from numcodecs.checksum32 import CRC32, Adler32, JenkinsLookup3


In [2]:
spotiphy_proj_dir = Path("spotiphy_project_files") / "Human Kidney Cancer Interface"

In [3]:
project_metadata = ProjectMetadata("hyperspot_project_files/Human Kidney Cancer Interface")


In [4]:
adata_ST = load_orig_st_data()

In [5]:
adata_aggr = load_aggr_st_data()
adata_aggr

AnnData object with n_obs × n_vars = 2035 × 10789
    obs: 'domain'
    var: 'SYMBOL'
    uns: 'domain', 'log1p', 'neighbors', 'spatial'
    obsm: 'embedding', 'spatial'
    obsp: 'connectivities', 'distances'

In [6]:
common_spots = adata_ST.obs_names.intersection(adata_aggr.obs_names)

In [7]:
adata_ST = adata_ST[common_spots]
adata_ST

View of AnnData object with n_obs × n_vars = 2035 × 13707
    obs: 'in_tissue', 'array_row', 'array_col', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mt_frac', 'donor_id', 'region', 'n_genes'
    var: 'feature_types', 'genome', 'SYMBOL', 'mt', 'n_cells_by_counts-6800STDY12499411', 'mean_counts-6800STDY12499411', 'log1p_mean_counts-6800STDY12499411', 'pct_dropout_by_counts-6800STDY12499411', 'total_counts-6800STDY12499411', 'log1p_total_counts-6800STDY12499411', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'hvg', 'log1p', 'sample_colors', 'spatial'
    obsm: 'spatial'

In [8]:
X_aggr = adata_aggr.X
X_aggr

array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  8.81551262],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  9.28929473],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  8.28898698],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  9.32384192],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  9.47205959],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        , 10.30162568]], shape=(2035, 10789))

In [9]:
X_st = adata_ST.X
X_st

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 5825041 stored elements and shape (2035, 13707)>

In [10]:
embedding = torch.load(project_metadata.project_folder / "hyperedge_embeddings.pth")

nan_mask = torch.isnan(embedding)
nan_rows = torch.any(nan_mask, dim=1)
embedding = embedding[~nan_rows]

In [11]:
embedding.numpy()

array([[ 8.889059  ,  8.679748  ,  4.4850726 , ...,  3.8536215 ,
        -3.6105201 ,  2.908497  ],
       [ 8.821315  ,  3.7601905 ,  1.8061496 , ...,  1.3176091 ,
        -2.5711322 ,  3.1535041 ],
       [11.43281   ,  4.174771  ,  4.087602  , ...,  1.1607624 ,
        -3.9088776 ,  2.4276586 ],
       ...,
       [12.548266  ,  3.1017194 ,  3.3315609 , ...,  2.203356  ,
        -3.4259186 ,  2.3580158 ],
       [12.063011  ,  4.230075  ,  2.2612758 , ...,  0.59771484,
        -3.5382173 ,  3.452862  ],
       [ 9.636461  ,  3.214257  ,  1.9453202 , ...,  2.2906444 ,
        -2.901063  ,  1.8127346 ]], shape=(2035, 200), dtype=float32)

In [12]:
for run_id in range(8):
    run_dir = project_metadata.eval_dir / f"run_{run_id}"

    st_clusters = pd.read_csv(run_dir / "clusters_ST.csv")
    st_svgs = pd.read_csv(run_dir / "svgs_ST.csv")

    ss_st = silhouette_score(X_st, st_clusters['domain'])
    ss_aggr = silhouette_score(X_aggr, st_clusters['domain'])
    ss_embedding = silhouette_score(embedding.numpy(), st_clusters['domain'])
    ss_dist = silhouette_score(adata_aggr.obsp['distances'], st_clusters['domain'], metric='precomputed')
    ss_svg = silhouette_score(adata_aggr[:,st_svgs['Unnamed: 0']].X, st_clusters['domain'])
    print(ss_st, ss_aggr, ss_embedding, ss_dist, ss_svg)

-0.003541304264217615 -0.014570240414086265 0.16907015442848206 -0.9995085995085995 -0.015447312815412964
-0.0065763420425355434 -0.017163207033465678 0.15785756707191467 -1.0 -0.020183024628254533
-0.008366814814507961 -0.01859890160827868 0.1385592669248581 -1.0 -0.028201568283358337
-0.005833420902490616 -0.015827678609395365 0.14597052335739136 -1.0 -0.02558177226784244
-0.00650011096149683 -0.017477772308542958 0.15980014204978943 -1.0 -0.021062726582833917
-0.007501556538045406 -0.01840433449669604 0.12869052588939667 -1.0 -0.023285666242492092
-0.006534349638968706 -0.01761466243980774 0.1444021761417389 -1.0 -0.019442891599586024
-0.007795078679919243 -0.017553061613448148 0.1466512531042099 -0.9995085995085995 -0.025142340746399542


In [13]:
silhouette_score(X_st, st_clusters['domain'])

-0.007795078679919243

In [14]:
sc.pp.pca(adata_ST, n_comps=50)
sc.pp.neighbors(adata_ST)
sc.tl.leiden(adata_ST, flavor='igraph', n_iterations=2)
leiden_orig = adata_ST.obs['leiden']
X_pca = adata_ST.obsm['X_pca']
silhouette_score(X_pca, leiden_orig)

/opt/conda/envs/HGCTSVG/lib/python3.12/site-packages/scanpy/preprocessing/_pca/__init__.py:384: ImplicitModificationWarning: Setting element `.obsm['X_pca']` of view, initializing view as actual.
  adata.obsm[key_obsm] = x_pca


0.12261264026165009

In [15]:
X_pca.shape

(2035, 50)

In [16]:
silhouette_samples(X_pca, adata_ST.obs['leiden'].to_numpy()).max(), silhouette_samples(X_pca, adata_ST.obs['leiden'].to_numpy()).min()

(np.float32(0.33518967), np.float32(-0.2427397))

In [17]:
silhouette_samples(X, st_clusters['leiden']).max(), silhouette_samples(X, st_clusters['leiden']).min()

NameError: name 'X' is not defined

In [ ]:
silhouette_samples(embedding.numpy(), st_clusters['leiden']).max(), silhouette_samples(embedding.numpy(), st_clusters['leiden']).min()

In [ ]:
# silhouette_samples(adata_aggr[:,st_svgs['Unnamed: 0']].X, st_clusters['leiden']).max()

In [ ]:
adata_aggr[:,st_svgs['Unnamed: 0']]

In [ ]:
adata_aggr.obsp['distances'].toarray()

In [ ]:
nz = adata_aggr.obsp['distances'].nonzero()
adata_aggr.obsp['distances'][nz]

In [ ]:
X_st.todense()

In [ ]:
st_clusters

In [ ]:
# import dbcv

# for run_id in range(8):
#     run_dir = project_metadata.eval_dir / f"run_{run_id}"

#     st_clusters = pd.read_csv(run_dir / "clusters_ST.csv")
#     st_svgs = pd.read_csv(run_dir / "svgs_ST.csv")

#     # ss_st = CDbw(X_st.todense(), st_clusters['leiden'])
#     # ss_aggr = CDbw(X_aggr, st_clusters['leiden'])
#     # dbcv_embedding = dbcv.dbcv(embedding.numpy(), st_clusters['leiden'])
#     # ss_dist = CDbw(adata_aggr.obsp['distances'], st_clusters['leiden'], metric='precomputed')

#     dbcv_st = dbcv.dbcv(X_st.todense(), st_clusters['leiden'], n_processes=40)
#     dbcv_aggr = dbcv.dbcv(X_aggr, st_clusters['leiden'], n_processes=40)
#     dbcv_svg = dbcv.dbcv(adata_aggr[:,st_svgs['Unnamed: 0']].X, st_clusters['leiden'], n_processes=40)
#     print(dbcv_st, dbcv_aggr, dbcv_svg)

In [ ]:
st_clusters['leiden'].to_numpy()

In [ ]:
from permetrics import ClusteringMetric

for run_id in range(8):
    run_dir = project_metadata.eval_dir / f"run_{run_id}"

    st_clusters = pd.read_csv(run_dir / "clusters_ST.csv")
    st_svgs = pd.read_csv(run_dir / "svgs_ST.csv")

    # ss_st = CDbw(X_st.todense(), st_clusters['leiden'])
    # ss_aggr = CDbw(X_aggr, st_clusters['leiden'])
    # dbcv_embedding = dbcv.dbcv(embedding.numpy(), st_clusters['leiden'])
    # ss_dist = CDbw(adata_aggr.obsp['distances'], st_clusters['leiden'], metric='precomputed')

    cm_st = ClusteringMetric(X=X_st.todense(), y_pred=st_clusters['leiden'].to_numpy())
    cm_aggr = ClusteringMetric(X=X_aggr, y_pred=st_clusters['leiden'].to_numpy())
    cm_svg = ClusteringMetric(X=adata_aggr[:,st_svgs['Unnamed: 0']].X, y_pred=st_clusters['leiden'].to_numpy())
    
    dbcv_st = cm_st.DBCVI()
    dbcv_aggr = cm_aggr.DBCVI()
    dbcv_svg = cm_svg.DBCVI()
    print(dbcv_st, dbcv_aggr, dbcv_svg)

In [ ]:
sc.pp.neighbors(adata_ST)
sc.tl.leiden(adata_ST)


In [ ]:
leiden_orig = adata_ST.obs['leiden'].to_numpy()
cm_orig = ClusteringMetric(X=X_st.todense(), y_pred=leiden_orig)
dbcv_orig = cm_orig.DBCVI()
print("Clustering score (leiden on PCA):", dbcv_orig)

dbcv_run_scores = []
for run_id in range(10):
    run_dir = project_metadata.eval_dir / f"run_{run_id}"

    st_clusters = pd.read_csv(run_dir / "clusters_ST.csv")
    # st_svgs = pd.read_csv(run_dir / "svgs_ST.csv")

    # ss_st = CDbw(X_st.todense(), st_clusters['leiden'])
    # ss_aggr = CDbw(X_aggr, st_clusters['leiden'])
    # dbcv_embedding = dbcv.dbcv(embedding.numpy(), st_clusters['leiden'])
    # ss_dist = CDbw(adata_aggr.obsp['distances'], st_clusters['leiden'], metric='precomputed')

    cm_embedding = ClusteringMetric(X=X_st.todense(), y_pred=st_clusters['leiden'].to_numpy())
    
    dbcv_embedding = cm_embedding.DBCVI()
    dbcv_run_scores.append(dbcv_embedding)
    print(f"run {run_id}:", dbcv_embedding)

print("Average score:", sum(dbcv_run_scores)/len(dbcv_run_scores))

In [18]:

sum([0.9841206305524953,
0.9839727454854618,
0.9839287958129314,
0.9838740729699287,
0.9851379290736353,
0.9832540170201871,
0.9832892099439836,
0.9839748129065214,
0.9840611894771564,
0.9846989212978124]) / 10

0.9840312324540113

In [ ]:
from hyperspot.utils import load_decomposed_st_data
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

# --- helper functions ---
def compute_metrics_orig(ct, adata_decomp):
    adata_ct = adata_decomp[adata_decomp.obs['cell_type'] == ct].copy()
    
    n_components = min(50, adata_ct.n_obs - 1, adata_ct.n_vars - 1)
    sc.pp.pca(adata_ct, n_comps=n_components)
    sc.pp.neighbors(adata_ct)
    sc.tl.leiden(adata_ct, flavor='igraph', n_iterations=2)
    leiden_ct_orig = adata_ct.obs['leiden'].to_numpy()
    
    X_pca = adata_ct.obsm['X_pca']
    cm_raw = ClusteringMetric(X=adata_ct.X, y_pred=leiden_ct_orig)
    cm_pca = ClusteringMetric(X=X_pca, y_pred=leiden_ct_orig)
    return {
        'silhouette': cm_pca.SI(),
        'dbcvi': cm_raw.DBCVI(),
        'rsi': cm_raw.RSI(),
    }

def compute_metrics_embedding(ct, run_dir, adata_decomp):
    adata_ct = adata_decomp[adata_decomp.obs['cell_type'] == ct].copy()
    ct_clusters = pd.read_csv(run_dir / f"clusters_ct_{ct}.csv")
    y_pred = ct_clusters['leiden'].to_numpy()
    
    n_components = min(50, adata_ct.n_obs - 1, adata_ct.n_vars - 1)
    sc.pp.pca(adata_ct, n_comps=n_components)
    X_pca = adata_ct.obsm['X_pca']
    
    cm_raw = ClusteringMetric(X=adata_ct.X, y_pred=y_pred)
    cm_pca = ClusteringMetric(X=X_pca, y_pred=y_pred)
    return {
        'silhouette': cm_pca.SI(),
        'dbcvi': cm_raw.DBCVI(),
        'rsi': cm_raw.RSI(),
    }

def mean_metric(results, key):
    return np.mean([r[key] for r in results])

# --- parallelized execution ---
adata_decomp = load_decomposed_st_data()

orig_results = Parallel(n_jobs=40)(
    delayed(compute_metrics_orig)(ct, adata_decomp) for ct in project_metadata.ct_list
)
print("Clustering score (leiden on PCA of original):")
print(f"  Silhouette: {mean_metric(orig_results, 'silhouette'):.4f}")
print(f"  DBCVI:      {mean_metric(orig_results, 'dbcvi'):.4f}")
print(f"  RSI:        {mean_metric(orig_results, 'rsi'):.4f}")

run_dirs = [project_metadata.eval_dir / f"run_{run_id}" for run_id in range(10)]
all_run_results = []
for run_dir in run_dirs:
    run_results = Parallel(n_jobs=40)(
        delayed(compute_metrics_embedding)(ct, run_dir, adata_decomp) for ct in project_metadata.ct_list
    )
    all_run_results.append(run_results)

print("\nClustering score (leiden on embedding, evaluated on original):")
for metric in ['silhouette', 'dbcvi', 'rsi']:
    per_run_means = [mean_metric(run_results, metric) for run_results in all_run_results]
    print(f"  {metric.capitalize():>12}: {np.mean(per_run_means):.4f} ± {np.std(per_run_means):.4f}")

In [ ]:
import numpy as np

dbcv_ct_origs = []
for ct in project_metadata.ct_list:
    adata_ct = load_ct_spatial_data(ct)
    sc.pp.neighbors(adata_ct)
    sc.tl.leiden(adata_ct)
    leiden_ct_orig = adata_ct.obs['leiden'].to_numpy()
    cm_ct_orig = ClusteringMetric(X=adata_ct.X, y_pred=leiden_ct_orig)
    dbcv_ct_orig = cm_ct_orig.DBCVI()
    dbcv_ct_origs.append(dbcv_ct_orig)

dbcv_ct_orig_mean = np.mean(dbcv_ct_origs)
print("Clustering score (leiden on PCA):", dbcv_ct_orig_mean)

all_ct_dbcvs = []
for run_id in range(10):
    run_dir = project_metadata.eval_dir / f"run_{run_id}"
    dbcv_ct_embeddings = []
    for ct in project_metadata.ct_list:
        adata_ct = load_ct_spatial_data(ct)
        ct_clusters = pd.read_csv(run_dir / f"clusters_ct_{ct}.csv")
        cm_embedding = ClusteringMetric(X=adata_ct.X, y_pred=ct_clusters['leiden'].to_numpy())
        ct_dbcv = cm_embedding.DBCVI()
        dbcv_ct_embeddings.append(ct_dbcv)
    run_ct_dbcv = np.mean(dbcv_ct_embeddings)  # Fixed: was dbcv_ct_embeddings.mean()
    all_ct_dbcvs.append(run_ct_dbcv)

all_ct_dbcvs_mean = np.mean(all_ct_dbcvs)  # Fixed: was all_ct_dbcvs.mean()
print("Average all runs:", all_ct_dbcvs_mean)

In [ ]:
for ct in project_metadata.ct_list:
    adata_ct = load_ct_spatial_data(ct)
    print(f"{ct}: {adata_ct.n_obs} cells, {adata_ct.n_vars} genes")

In [ ]:
from hyperspot.utils import load_decomposed_st_data
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.metrics import silhouette_score

# --- helper functions ---
def compute_silhouette_orig(ct, adata_decomp):
    adata_ct = adata_decomp[adata_decomp.obs['cell_type'] == ct].copy()
    
    n_components = min(50, adata_ct.n_obs - 1, adata_ct.n_vars - 1)
    sc.pp.pca(adata_ct, n_comps=n_components)
    sc.pp.neighbors(adata_ct)
    sc.tl.leiden(adata_ct)
    leiden_ct_orig = adata_ct.obs['leiden'].to_numpy()
    
    X_pca = adata_ct.obsm['X_pca']
    return silhouette_score(X_pca, leiden_ct_orig)

def compute_silhouette_embedding(ct, run_dir, adata_decomp):
    adata_ct = adata_decomp[adata_decomp.obs['cell_type'] == ct].copy()
    ct_clusters = pd.read_csv(run_dir / f"clusters_ct_{ct}.csv")
    leiden_labels = ct_clusters['leiden'].to_numpy()

    # PCA before silhouette — important at 200d
    n_components = min(50, adata_ct.n_obs - 1, adata_ct.n_vars - 1)
    sc.pp.pca(adata_ct, n_comps=n_components)
    X_pca = adata_ct.obsm['X_pca']

    return silhouette_score(X_pca, leiden_labels)

# --- parallelized execution ---
adata_decomp = load_decomposed_st_data()
silhouette_ct_origs = Parallel(n_jobs=40)(
    delayed(compute_silhouette_orig)(ct, adata_decomp) for ct in project_metadata.ct_list
)
silhouette_ct_orig_mean = np.mean(silhouette_ct_origs)
print("Clustering score (leiden on PCA):", silhouette_ct_orig_mean)

run_dirs = [project_metadata.eval_dir / f"run_{run_id}" for run_id in range(10)]
all_ct_silhouettes = []
for run_dir in run_dirs:
    silhouette_ct_embeddings = Parallel(n_jobs=40)(
        delayed(compute_silhouette_embedding)(ct, run_dir, adata_decomp) for ct in project_metadata.ct_list
    )
    all_ct_silhouettes.append(np.mean(silhouette_ct_embeddings))

all_ct_silhouettes_mean = np.mean(all_ct_silhouettes)
print("Average all runs:", all_ct_silhouettes_mean)

In [ ]:
from hyperspot.utils import load_decomposed_st_data
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

# --- helper functions ---

def compute_dbcv_orig(ct, adata_decomp):
    adata_ct = adata_decomp[adata_decomp.obs['cell_type'] == ct].copy()
    
    n_components = min(50, adata_ct.n_obs - 1, adata_ct.n_vars - 1)
    sc.pp.pca(adata_ct, n_comps=n_components)
    sc.pp.neighbors(adata_ct)
    sc.tl.leiden(adata_ct)
    leiden_ct_orig = adata_ct.obs['leiden'].to_numpy()
    cm = ClusteringMetric(X=adata_ct.X, y_pred=leiden_ct_orig)
    return cm.RSI()

def compute_dbcv_embedding(ct, run_dir, adata_decomp):
    adata_ct = adata_decomp[adata_decomp.obs['cell_type'] == ct].copy()
    ct_clusters = pd.read_csv(run_dir / f"clusters_ct_{ct}.csv")
    cm = ClusteringMetric(X=adata_ct.X, y_pred=ct_clusters['leiden'].to_numpy())
    return cm.RSI()

# --- parallelized execution ---

adata_decomp = load_decomposed_st_data()

dbcv_ct_origs = Parallel(n_jobs=40)(
    delayed(compute_dbcv_orig)(ct, adata_decomp) for ct in project_metadata.ct_list
)
dbcv_ct_orig_mean = np.mean(dbcv_ct_origs)
print("Clustering score (leiden on PCA):", dbcv_ct_orig_mean)

run_dirs = [project_metadata.eval_dir / f"run_{run_id}" for run_id in range(10)]

all_ct_dbcvs = []
for run_dir in run_dirs:
    dbcv_ct_embeddings = Parallel(n_jobs=40)(
        delayed(compute_dbcv_embedding)(ct, run_dir, adata_decomp) for ct in project_metadata.ct_list
    )
    all_ct_dbcvs.append(np.mean(dbcv_ct_embeddings))

all_ct_dbcvs_mean = np.mean(all_ct_dbcvs)
print("Average all runs:", all_ct_dbcvs_mean)



In [ ]:
from hyperspot.utils import load_decomposed_st_data
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

# --- helper functions ---

def compute_dbi_orig(ct, adata_decomp):
    adata_ct = adata_decomp[adata_decomp.obs['cell_type'] == ct].copy()
    
    n_components = min(50, adata_ct.n_obs - 1, adata_ct.n_vars - 1)
    sc.pp.pca(adata_ct, n_comps=n_components)
    sc.pp.neighbors(adata_ct)
    sc.tl.leiden(adata_ct)
    leiden_ct_orig = adata_ct.obs['leiden'].to_numpy()
    cm = ClusteringMetric(X=adata_ct.X, y_pred=leiden_ct_orig)
    return cm.DBI()

def compute_dbi_embedding(ct, run_dir, adata_decomp):
    adata_ct = adata_decomp[adata_decomp.obs['cell_type'] == ct].copy()
    ct_clusters = pd.read_csv(run_dir / f"clusters_ct_{ct}.csv")
    cm = ClusteringMetric(X=adata_ct.X, y_pred=ct_clusters['leiden'].to_numpy())
    return cm.DBI()

# --- parallelized execution ---

adata_decomp = load_decomposed_st_data()

dbcv_ct_origs = Parallel(n_jobs=40)(
    delayed(compute_dbi_orig)(ct, adata_decomp) for ct in project_metadata.ct_list
)
dbcv_ct_orig_mean = np.mean(dbcv_ct_origs)
print("Clustering score (leiden on PCA):", dbcv_ct_orig_mean)

run_dirs = [project_metadata.eval_dir / f"run_{run_id}" for run_id in range(10)]

all_ct_dbcvs = []
for run_dir in run_dirs:
    dbcv_ct_embeddings = Parallel(n_jobs=40)(
        delayed(compute_dbi_embedding)(ct, run_dir, adata_decomp) for ct in project_metadata.ct_list
    )
    all_ct_dbcvs.append(np.mean(dbcv_ct_embeddings))

all_ct_dbcvs_mean = np.mean(all_ct_dbcvs)
print("Average all runs:", all_ct_dbcvs_mean)



In [ ]:


run_dir = project_metadata.eval_dir / f"run_{run_id}"
svgs_ST = pd.read_csv(run_dir / "svgs_ST.csv")['Unnamed: 0']
set(svgs_ST)

In [ ]:
from itertools import combinations
def svg_stability_report():
    runs = []
    lengths = []
    for run_id in range(10):
        run_dir = project_metadata.eval_dir / f"run_{run_id}"
        svgs_ST = pd.read_csv(run_dir / "svgs_ST.csv")['Unnamed: 0']
        runs.append(set(svgs_ST))
        lengths.append(len(svgs_ST))

    # pairwise containment
    scores = []
    for a, b in combinations(runs, 2):
        scores.append(len(a & b) / min(len(a), len(b)))
    
    # stable core: items in >80% of runs
    all_items = set().union(*runs)
    stable_core = {item for item in all_items
                   if sum(item in s for s in runs) / len(runs) >= 0.8}
    
    return {
        "stability_score": sum(scores) / len(scores),
        "score_std": np.std(scores),
        "length_mean": np.mean(lengths),
        "length_std": np.std(lengths),
        "stable_core": stable_core,
        "unstable_items": all_items - stable_core,
    }

svg_stability_report()

In [ ]:
embedding.shape

In [ ]:
from itertools import combinations

def ct_svg_stability_report():
    runs = []
    lengths = []
    for run_id in range(10):
        svgs = set()
        for ct in project_metadata.ct_list:
            ct_run_dir = project_metadata.eval_dir / f"run_{run_id}"
            # print(ct_run_dir)
            svgs_ct = pd.read_csv(run_dir / f"svgs_ct_{ct}.csv")['Unnamed: 0']
            svgs = svgs | set(svgs_ct)
            # print(ct, len(svgs_ct), len(svgs))

        # print()
        runs.append(svgs)
        lengths.append(len(svgs))

    # pairwise containment
    scores = []
    for a, b in combinations(runs, 2):
        scores.append(len(a & b) / min(len(a), len(b)))

    print([len(run) for run in runs])
    print(len(scores))
    print(sum(scores))
    stability_score = sum(scores)/len(scores)
    # ct_score_std = np.std(scores)
    # ct_stability_scores.append(ct_stability_score)
    
    # run_ct_stability_scores.append(np.mean(ct_stability_scores))

    return stability_score
    # return np.mean(run_ct_stability_scores)
ct_svg_stability_report()

In [ ]:
import squidpy as sq
def svg_fdr_report():
    run_id = 9

    significant_genes = []
    adata_aggr = load_aggr_st_data()

    run_dir = project_metadata.eval_dir / f"run_{run_id}"
    svgs = pd.read_csv(run_dir / f"svgs_ST.csv")['Unnamed: 0']

    sq.gr.spatial_neighbors(adata_aggr, key_added='spatial')
    sq.gr.spatial_autocorr(
        adata_aggr,
        mode="moran",
        genes = svgs,
        n_perms=1000
    )

    moran_df = adata_ct.uns["moranI"]

    fdr_thresh = 0.05

    print(moran_df.keys())
    significant = (moran_df["pval_norm_fdr_bh"] < fdr_thresh)
    # significant = (moran_df["pval_sim_fdr_bh"] < fdr_thresh) & (moran_df["I"] > 0)
    percent_significant = significant.mean() * 100
    
    print(f"{ct}: {percent_significant:.2f}% of genes pass FDR < {fdr_thresh}")
        
    # return np.mean(run_ct_stability_scores)
svg_fdr_report()

In [ ]:
import squidpy as sq
def ct_svg_fdr_report():
    run_id = 9

    significant_genes = []
    for ct in project_metadata.ct_list:
        adata_ct = load_ct_spatial_data(ct)
        ct_run_dir = project_metadata.eval_dir / f"run_{run_id}"
        svgs_ct = pd.read_csv(run_dir / f"svgs_ct_{ct}.csv")['Unnamed: 0']

        if len(svgs_ct) == 0:
            continue
            
        sq.gr.spatial_autocorr(
            adata_ct,
            mode="moran",
            genes = svgs_ct,
            n_perms=100
        )

        moran_df = adata_ct.uns["moranI"]

        fdr_thresh = 0.01

        significant = (moran_df["pval_sim_fdr_bh"] < fdr_thresh)
        # significant = (moran_df["pval_sim_fdr_bh"] < fdr_thresh) & (moran_df["I"] > 0)
        percent_significant = significant.mean() * 100
        
        print(f"{ct}: {percent_significant:.2f}% of genes pass FDR < {fdr_thresh}")
        significant_genes.append(percent_significant)

    print("average across all cell types:", np.mean(significant_genes))
        
    # return np.mean(run_ct_stability_scores)
ct_svg_fdr_report()

In [ ]:
for run_id in range(8):
    run_dir = project_metadata.eval_dir / f"run_{run_id}"

    st_clusters = pd.read_csv(run_dir / "clusters_ST.csv")
    st_svgs = pd.read_csv(run_dir / "svgs_ST.csv")

    print(silhouette_score(embedding.numpy(), st_clusters['leiden']))

In [ ]:
st_clusters['leiden']

In [ ]:
X.shape

In [ ]:
import squidpy as sq
def sepal_fdr_report():
    run_id = 9

    adata_decomp = load_decomposed_st_data()
    
    significant_genes = []
    for ct in project_metadata.ct_list:
        adata_ct = load_ct_spatial_data(ct)
        ct_run_dir = project_metadata.eval_dir / f"run_{run_id}"
        svgs_ct = pd.read_csv(run_dir / f"svgs_ct_{ct}.csv")['Unnamed: 0']

        if len(svgs_ct) == 0:w
            continue
            
        sq.gr.spatial_autocorr(
            adata_ct,
            mode="moran",
            genes = svgs_ct,
            n_perms=100
        )

        moran_df = adata_ct.uns["moranI"]

        fdr_thresh = 0.05

        significant = (moran_df["pval_sim_fdr_bh"] < fdr_thresh)
        # significant = (moran_df["pval_sim_fdr_bh"] < fdr_thresh) & (moran_df["I"] > 0)
        percent_significant = significant.mean() * 100
        
        print(f"{ct}: {percent_significant:.2f}% of genes pass FDR < {fdr_thresh}")
        significant_genes.append(percent_significant)

    print("average across all cell types:", np.mean(significant_genes))
        
    # return np.mean(run_ct_stability_scores)
ct_svg_fdr_report()